# 03 · WASDE (USDA)

Tabela *World Soybean Supply and Use* de cada relatório mensal, baixada do ESMIS. O primeiro download leva alguns minutos; depois fica no cache `data/wasde_tabelas.csv`.

In [ ]:
import sys
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, features, wasde

ETAPA = "03_wasde"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})

## 1. Relatórios

In [ ]:
# se o cache veio junto com o projeto (data/ no Dataset), uso ele; senão baixo tudo
cache = RAIZ / "data" / "wasde_tabelas.csv"
if not cache.exists():
    cache = config.DATA / "wasde_tabelas.csv"
tabelas = wasde.carregar_tabelas(cache, desde=config.INICIO)

por_ano = pd.PeriodIndex(tabelas["relatorio"].unique(), freq="M").year.value_counts().sort_index()
por_ano.plot.bar(color="tab:blue", rot=90, title="Relatórios lidos por ano (o normal é 12)")
plt.axhline(12, color="k", ls="--", lw=.8); plt.show()
print("Cache em", cache)

## 2. Features de oferta

- `estoque_uso_mundo` e `estoque_uso_eua`: estoque final ÷ consumo total (baixo = mercado apertado)
- `revisao_brasil`, `revisao_eua`: ln(produção neste relatório ÷ no anterior)
- `revisao_china_import`: o mesmo para as importações da China

In [ ]:
oferta = features.features_wasde(tabelas).loc[config.INICIO:]
print("Meses sem relatório:", list(oferta.index[oferta["sem_relatorio"]].astype(str)))

fig, ax = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
cores = ["tab:blue", "tab:purple", "tab:brown", "tab:red", "tab:olive"]
for a, coluna, cor in zip(ax, features.ESTOQUES + features.REVISOES, cores):
    oferta[coluna].plot(ax=a, color=cor, title=coluna)
for a in ax[2:]:
    a.axhline(0, c="k", lw=.7)
fig.tight_layout(); fig.savefig(PASTA / "wasde.png", dpi=120); plt.show()
oferta.tail()

## 3. Correlação com o alvo (antes do teste)

In [ ]:
preco = pd.read_csv(config.ler_resultado("01_preco", "preco.csv"), index_col="mes")
preco.index = pd.PeriodIndex(preco.index, freq="M")

juntos = oferta.drop(columns="sem_relatorio").join(preco[["alvo", "dlog_preco"]], how="inner")
juntos = juntos.loc[:pd.Period(config.TESTE_INICIO) - 1]
corr = pd.DataFrame({
    "alvo (mês seguinte)": juntos[features.ESTOQUES + features.REVISOES].corrwith(juntos["alvo"]),
    "variação do mesmo mês": juntos[features.ESTOQUES + features.REVISOES].corrwith(juntos["dlog_preco"]),
})
corr.plot.bar(rot=0, title="Correlação das features de oferta"); plt.axhline(0, color="k", lw=.8); plt.show()
corr.round(3)

## 4. Salvando

In [ ]:
saida = oferta.copy()
saida.index = saida.index.astype(str)
saida.index.name = "mes"
saida.to_csv(PASTA / "wasde.csv")
config.salvar_config(
    PASTA, fonte="USDA WASDE via ESMIS", relatorios=int(tabelas["relatorio"].nunique()),
    meses=len(saida), ultimo_mes=saida.index[-1],
    meses_sem_relatorio=list(saida.index[saida["sem_relatorio"]]),
)
print(sorted(p.name for p in PASTA.iterdir()))